<a href="https://colab.research.google.com/github/loukikjoshi06-ops/NIRVANA-gpt/blob/main/Copy_of_NIRVANA_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F



In [2]:
import os

print(os.path.exists("/content/checkpoint_latest.pth"))

False


In [3]:
from google.colab import files

uploaded = files.upload()

Saving TinyStories-small.txt to TinyStories-small.txt


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
checkpoint = torch.load(
    "checkpoint_latest.pth",
    map_location=device
)

NameError: name 'device' is not defined

In [ ]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

In [ ]:
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

In [ ]:
start_iter = checkpoint["iteration"]

here dataset is uploaded

In [4]:
with open('TinyStories-small.txt','r', encoding='utf-8')as l:
    text = l.read()

In [5]:
print(text[ :100])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with


In [6]:
char = sorted(list(set(text)))
print (len(char))

106


In [7]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}


In [8]:
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])

In [9]:
test = "loukik"
print(encode(test))
print(decode(encode(test)))

[71, 74, 80, 70, 68, 70]
loukik


In [10]:
data =torch.tensor(encode(text),dtype = torch.long)

In [11]:
print(data.shape)
print(data[:50])

torch.Size([52391770])
tensor([45, 73, 64,  2, 63, 60, 84, 12,  2, 60,  2, 71, 68, 79, 79, 71, 64,  2,
        66, 68, 77, 71,  2, 73, 60, 72, 64, 63,  2, 42, 68, 71, 84,  2, 65, 74,
        80, 73, 63,  2, 60,  2, 73, 64, 64, 63, 71, 64,  2, 68])


In [12]:
n = int(0.8 * len(data))
train_data = data[ :n]
val_data = data [n: ]

In [13]:
batch_size = 32
block_size = 64
vocab_size = len(char)

In [14]:
batch_size = 64
block_size = 256
max_iters = 10000
eval_interval = 500
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2


In [15]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [16]:
xb,yb = get_batch('train')
print(xb.shape)
print(yb.shape)

torch.Size([64, 256])
torch.Size([64, 256])


In [17]:
for b in range(1):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]
    print(context.tolist(),"-->",target.item())

[66] --> 78
[66, 78] --> 2
[66, 78, 2] --> 68
[66, 78, 2, 68] --> 73
[66, 78, 2, 68, 73] --> 2
[66, 78, 2, 68, 73, 2] --> 79
[66, 78, 2, 68, 73, 2, 79] --> 67
[66, 78, 2, 68, 73, 2, 79, 67] --> 64
[66, 78, 2, 68, 73, 2, 79, 67, 64] --> 2
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2] --> 72
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72] --> 74
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74] --> 77
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77] --> 73
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73] --> 68
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73, 68] --> 73
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73, 68, 73] --> 66
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73, 68, 73, 66] --> 12
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73, 68, 73, 66, 12] --> 2
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73, 68, 73, 66, 12, 2] --> 68
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 73, 68, 73, 66, 12, 2, 68] --> 73
[66, 78, 2, 68, 73, 2, 79, 67, 64, 2, 72, 74, 77, 

In [18]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()

    self.key = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)

    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self,x):

    B, T, C = x.shape

    k = self.key(x)
    q = self.query(x)

    head_size = k.shape[-1]


    wei = q @ k.transpose(-2, -1)
    wei = wei * head_size**-0.5

    wei = wei.masked_fill(self.tril[:T , :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim = -1)
    wei = self.dropout(wei)
    v = self.value(x)
    out = wei @ v

    return out




In [19]:
class MultiHeadAttention(nn.Module):

  def __init__(self,num_heads,head_size):
    super().__init__()

    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    self.proj = nn.Linear(head_size * num_heads, n_embd)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
     out = torch.cat([h(x) for h in self.heads], dim = -1)
     out = self.proj(out)
     out = self.dropout(out)

     return out

In [20]:
class FeedForward(nn.Module):

  def __init__(self,n_embd):
    super().__init__()

    self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd),
                             nn.ReLU(),
                             nn.Linear(4 * n_embd, n_embd),
                             nn.Dropout(dropout),)
  def forward(self,x):
        return self.net(x)

In [21]:
class Block(nn.Module):

  def __init__(self,n_embd,n_head):
    super().__init__()

    head_size = n_embd // n_head

    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
      x = x + self.sa(self.ln1(x))
      x = x + self.ffwd(self.ln2(x))

      return x

In [22]:
class NIRVANAgpt (nn.Module):
  def __init__(self):
   super().__init__()

   self.token_embedding_table = nn.Embedding (vocab_size,n_embd)

   self.position_embedding_table = nn.Embedding(block_size, n_embd)


   self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

   self.ln_f = nn.LayerNorm(n_embd)

   self.lm_head = nn.Linear(n_embd, vocab_size)


  def forward(self,idx,target = None):

    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx)

    pos_emb = self.position_embedding_table(torch.arange(T, device=device))


    x = tok_emb + pos_emb

    x = self.blocks(x)

    x = self.ln_f(x)

    logits = self.lm_head(x)

    if target is None:
      loss = None
      return logits, loss

    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = target.view(B*T)

      loss = F.cross_entropy(logits,targets)
      return logits,loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]

      logits, loss = self(idx_cond)

      logits = logits[:,-1,:]

      probs = F.softmax(logits,dim=-1)

      idx_next = torch.multinomial(probs,num_samples=1)

      idx = torch.cat((idx,idx_next),dim=1)

    return idx

In [23]:
model = NIRVANAgpt ().to(device)

xb,yb = get_batch('train')

logits,loss = model(xb,yb)
print("Logit shape : ", logits.shape)
print("Loss : ", loss)

Logit shape :  torch.Size([16384, 106])
Loss :  tensor(4.8491, device='cuda:0', grad_fn=<NllLossBackward0>)


In [24]:
print(device)

cuda


In [25]:
model = NIRVANAgpt ().to(device)



optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


model.train()

for iter in range(max_iters):


  xb,yb = get_batch('train')

  logits,loss = model(xb,yb)

  optimizer.zero_grad(set_to_none = True)

  loss.backward()

  optimizer.step()

  if iter % 100 == 0:
    print(f"Step {iter}: Loss = {loss.item():.4f}")



Step 0: Loss = 4.8297
Step 100: Loss = 2.3168
Step 200: Loss = 2.2330
Step 300: Loss = 2.0181
Step 400: Loss = 1.7720
Step 500: Loss = 1.5941
Step 600: Loss = 1.5183
Step 700: Loss = 1.4434
Step 800: Loss = 1.3760
Step 900: Loss = 1.3203
Step 1000: Loss = 1.2738
Step 1100: Loss = 1.2104
Step 1200: Loss = 1.1569
Step 1300: Loss = 1.1489
Step 1400: Loss = 1.1411
Step 1500: Loss = 1.0794
Step 1600: Loss = 1.0917
Step 1700: Loss = 1.0619
Step 1800: Loss = 1.0661
Step 1900: Loss = 1.0551
Step 2000: Loss = 1.0487
Step 2100: Loss = 1.0486
Step 2200: Loss = 0.9940
Step 2300: Loss = 1.0223
Step 2400: Loss = 0.9618
Step 2500: Loss = 0.9698
Step 2600: Loss = 0.9830
Step 2700: Loss = 0.9580
Step 2800: Loss = 0.9594
Step 2900: Loss = 0.9635
Step 3000: Loss = 0.9303
Step 3100: Loss = 0.9355
Step 3200: Loss = 0.9050
Step 3300: Loss = 0.9077
Step 3400: Loss = 0.9170
Step 3500: Loss = 0.8959
Step 3600: Loss = 0.8859
Step 3700: Loss = 0.9064
Step 3800: Loss = 0.8950
Step 3900: Loss = 0.8988
Step 4000: L

In [31]:
model.eval()

context = torch.zeros((1,1),dtype=torch.long,device=device)

print(decode(model.generate(context,max_new_tokens=500)[0].tolist()))

	laying it was hard because it could part it so much that it could find it. Every day the stranger would spread it, but it could not be alone. 
One while Lucy was patient and sparkly had done such a problem. She had big spot to fix it with her friends. 
After going on, the king tried to carry it back into the hambulance. Her friendship was about to fix it. She looked around for a while but she couldn't find it. Suddenly, she heard someone about the beach. She felt so happy! The king appeared the 


In [32]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "iteration": iter,
    "loss": loss.item(),
}, "/content/checkpoint_latest.pth")

In [33]:
import os

print(os.path.exists("/content/checkpoint_latest.pth"))

True


In [34]:
import os

size = os.path.getsize("/content/checkpoint_latest.pth") / (1024 * 1024)
print(f"{size:.2f} MB")

133.05 MB


In [35]:
from google.colab import files

files.download("/content/checkpoint_latest.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>